# Reproducible Preparation and Data Quality
## tl;dr
The revised pipeline starts from the original CSV rather than the Excel-limited coursework subset. It removes exact duplicates, retains unknown predictor values, and audits invalid targets, identifiers, dates, and vehicle years. It does not delete records simply because search, location, or demographic fields are missing.

## Context & Methods
### Key Assumptions
Exact-row deduplication uses hashes of all original columns. Repeated `SeqID` values are retained here because one stop may contain multiple violations. Invalid vehicle years become missing; numeric imputation is fitted only on model-training data. Missingness mechanisms are not labeled MCAR or MAR without evidence.

The original cleaned CSV is not an input to this pipeline. Its row count and undocumented historical selections are not treated as the full source population.

In [1]:
from pathlib import Path
import sys
ROOT = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
             if (candidate / "src/traffic_analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the traffic project repository.")
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 15)

## Data
This cell rebuilds the local prepared CSV and an aggregate audit. It can take a minute because the source file is large. Generated row-level data is ignored by Git.

In [2]:
from src.traffic_analysis import prepare, load_prepared
preparation_audit = prepare()
display(pd.Series(preparation_audit, name="value").to_frame())
data = load_prepared()

,value
raw_rows,1951063
exact_duplicates_removed,2276
invalid_target_rows,0
missing_stop_id_rows,0
invalid_date_rows,0
...,...
input_file,Traffic_Violations.csv
input_bytes,829632172
source_url,https://catalog.data.gov/dataset/traffic-viola...
grain,violation record; multiple records may share a...


## Results
Inspect missing predictors without converting unknowns to observed zeroes. The invalid-date count is reported separately; those records cannot enter a chronological evaluation.

In [3]:
display(data.isna().sum().sort_values(ascending=False).rename("missing_records").to_frame())
display(data.groupby("accident").size().rename(index={0:"No accident",1:"Accident"}).rename("records"))
assert len(data) == preparation_audit["prepared_records"]
assert data.accident.isin([0, 1]).all()
assert data.stop_id.notna().all()

,missing_records
vehicle_age,13072
stop_id,0
stop_date,0
accident,0
hour,0
weekday,0
month,0
subagency,0
vehicle_type,0
alcohol,0


accident
No accident    1895392
Accident         53395
Name: records, dtype: int64

## Takeaways
The audit records the input filename, size, SHA-256, observed dates, retained counts, and exclusions. These identify the actual snapshot used. Fresh downloads may differ. Notebook 3 uses violation records for exploration; notebook 4 uses one consistent-labeled record per stop and keeps all stop IDs out of overlapping partitions.